<a href="https://colab.research.google.com/github/Wo0ond3r/portfolio-tsuruoka-lab/blob/main/phase0-math-python/03-mlp-mnist/notebook/03_mlp_mnist.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 구현 과제 3 — MLP MNIST: NumPy → PyTorch → Lightning

> **Phase 0 — 수학 & Python 기초** | 구현 과제 3 / 3  ← **Phase 0 최종 과제**

---

## 이 노트북의 구조

같은 MLP를 **세 번** 구현합니다. 매번 코드가 짧아지고 추상화가 올라갑니다.

```
Part 1 — NumPy only     : 행렬 곱셈 + 역전파를 직접 구현 (과제 1·2 복습)
Part 2 — PyTorch        : autograd로 역전파 자동화, 학습 루프 직접 작성
Part 3 — Lightning      : 학습 루프까지 추상화, AdamW + 스케줄러 적용
```

## 학습 목표
- [ ] 행렬 곱셈이 신경망 레이어가 되는 과정을 직접 구현으로 확인
- [ ] `nn.Module` 구조와 학습 루프의 관계 이해
- [ ] AdamW + CosineAnnealingLR (LLM 표준 패턴) 적용
- [ ] wandb 하이퍼파라미터 스윕으로 최적 설정 탐색
- [ ] 세 구현의 결과가 동일함을 검증

## Phase 0에서 배운 것들의 연결
| 과제 1 | 과제 2 | 과제 3 |
|---|---|---|
| 행렬 곱셈 `A @ B` | `Value._backward()` | `nn.Linear` (내부가 `A @ B`) |
| 브로드캐스팅 `+ b` | `backward()` 위상 정렬 | `loss.backward()` |
| 코사인 유사도 | `MLP.zero_grad()` | `optimizer.zero_grad()` |

---
## 0. 환경 설정

In [ ]:
!pip install wandb pytorch-lightning -q

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets, transforms
import pytorch_lightning as pl
import wandb
import matplotlib.pyplot as plt
import random

# 재현성
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'디바이스: {DEVICE}')
print('환경 설정 완료 ✓')

In [ ]:
wandb.login()

run = wandb.init(
    project='portfolio-tsuruoka-lab',
    name='phase0-03-mlp-mnist',
    tags=['phase0', 'mnist', 'mlp', 'numpy', 'pytorch', 'lightning'],
    config={'phase': 0, 'assignment': 3, 'dataset': 'MNIST'}
)
print(f'wandb run: {run.url}')

---
## 1. 데이터 준비 — MNIST

MNIST: 손글씨 숫자 0~9, 28×28 픽셀, 총 70000장 (train 60000 / test 10000)

In [ ]:
# 데이터 다운로드 & 전처리
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))  # MNIST 평균/표준편차
])

train_dataset = datasets.MNIST('/tmp/mnist', train=True,  download=True, transform=transform)
test_dataset  = datasets.MNIST('/tmp/mnist', train=False, download=True, transform=transform)

print(f'train: {len(train_dataset)}장, test: {len(test_dataset)}장')
print(f'이미지 shape: {train_dataset[0][0].shape}  ← (채널, 높이, 너비)')

# 샘플 시각화
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    img, label = train_dataset[i]
    ax.imshow(img.squeeze(), cmap='gray')
    ax.set_title(str(label), fontsize=10)
    ax.axis('off')
plt.suptitle('MNIST 샘플 (16장)', fontsize=12)
plt.tight_layout()
wandb.log({'data/sample_images': wandb.Image(fig)})
plt.show()
print('샘플 wandb 업로드 ✓')

---
## Part 1 — NumPy Only MLP

과제 1·2에서 배운 것을 전부 연결합니다.

```
순전파: X → [W1, b1] → ReLU → [W2, b2] → Softmax → Loss
역전파: 연쇄법칙으로 ∂L/∂W2, ∂L/∂W1 계산
```

### 수식 연결
- **과제 1**: `Z = X @ W + b` ← 행렬 곱셈 + 브로드캐스팅
- **과제 2**: `dW = X.T @ dZ` ← 행렬 버전 연쇄법칙 (∂L/∂W = Xᵀ @ ∂L/∂Z)

In [ ]:
# NumPy 데이터 준비 (작은 subset으로 빠르게 실험)
N_TRAIN = 10000  # 전체 60000 중 10000만 사용 (속도)

X_train_np = train_dataset.data[:N_TRAIN].numpy().reshape(N_TRAIN, -1) / 255.0
y_train_np = train_dataset.targets[:N_TRAIN].numpy()
X_test_np  = test_dataset.data.numpy().reshape(-1, 784) / 255.0
y_test_np  = test_dataset.targets.numpy()

print(f'X_train: {X_train_np.shape}  ← (샘플 수, 784픽셀)')
print(f'y_train: {y_train_np.shape}  ← 0~9 레이블')

In [ ]:
class NumpyMLP:
    """
    NumPy로 구현한 2층 MLP.
    784 → 256 → 128 → 10

    과제 1: @ 연산자 = 행렬 곱셈
    과제 1: + b       = 브로드캐스팅
    과제 2: 연쇄법칙  = _backward 수동 구현
    """
    def __init__(self, lr=0.01):
        self.lr = lr
        # He 초기화: ReLU에 최적화된 가중치 초기화
        self.W1 = np.random.randn(784, 256) * np.sqrt(2/784)
        self.b1 = np.zeros(256)
        self.W2 = np.random.randn(256, 128) * np.sqrt(2/256)
        self.b2 = np.zeros(128)
        self.W3 = np.random.randn(128, 10)  * np.sqrt(2/128)
        self.b3 = np.zeros(10)

    def relu(self, z):
        return np.maximum(0, z)   # 과제 2에서 만든 relu 와 동일

    def softmax(self, z):
        # 수치 안정성: 최댓값 빼기 (브로드캐스팅!)
        z = z - z.max(axis=1, keepdims=True)
        exp_z = np.exp(z)
        return exp_z / exp_z.sum(axis=1, keepdims=True)

    def forward(self, X):
        # ── 순전파 ────────────────────────────────────────────────────
        # 과제 1: Z = X @ W + b  (행렬 곱셈 + 브로드캐스팅)
        self.X   = X
        self.Z1  = X  @ self.W1 + self.b1   # (N, 784) @ (784,256) = (N,256)
        self.A1  = self.relu(self.Z1)        # (N, 256)
        self.Z2  = self.A1 @ self.W2 + self.b2  # (N,256) @ (256,128) = (N,128)
        self.A2  = self.relu(self.Z2)        # (N, 128)
        self.Z3  = self.A2 @ self.W3 + self.b3  # (N,128) @ (128,10) = (N,10)
        self.out = self.softmax(self.Z3)     # (N, 10)
        return self.out

    def loss(self, y):
        # 크로스 엔트로피: -log(정답 클래스 확률)
        N = len(y)
        return -np.log(self.out[np.arange(N), y] + 1e-8).mean()

    def backward(self, y):
        # ── 역전파 (연쇄법칙 수동 적용) ──────────────────────────────
        N = len(y)

        # Softmax + CrossEntropy 합산 미분: out - one_hot(y)
        dZ3 = self.out.copy()
        dZ3[np.arange(N), y] -= 1
        dZ3 /= N

        # 3층 역전파: ∂L/∂W3 = A2ᵀ @ dZ3  (과제 2, 퀴즈 6번 내용!)
        dW3 = self.A2.T @ dZ3           # (128, 10)
        db3 = dZ3.sum(axis=0)           # (10,)   브로드캐스팅 역전파
        dA2 = dZ3 @ self.W3.T           # (N, 128)

        # ReLU 역전파: 양수만 통과 (퀴즈 2번 내용!)
        dZ2 = dA2 * (self.Z2 > 0)
        dW2 = self.A1.T @ dZ2
        db2 = dZ2.sum(axis=0)
        dA1 = dZ2 @ self.W2.T

        dZ1 = dA1 * (self.Z1 > 0)
        dW1 = self.X.T @ dZ1
        db1 = dZ1.sum(axis=0)

        # SGD 업데이트: w ← w - lr × ∂L/∂w
        self.W3 -= self.lr * dW3;  self.b3 -= self.lr * db3
        self.W2 -= self.lr * dW2;  self.b2 -= self.lr * db2
        self.W1 -= self.lr * dW1;  self.b1 -= self.lr * db1

    def accuracy(self, X, y):
        preds = self.forward(X).argmax(axis=1)
        return (preds == y).mean()


print('NumpyMLP 정의 완료 ✓')

In [ ]:
# NumPy MLP 학습
np.random.seed(SEED)
model_np = NumpyMLP(lr=0.05)

BATCH = 256
EPOCHS_NP = 20
np_losses, np_accs = [], []

for epoch in range(EPOCHS_NP):
    # 셔플
    idx = np.random.permutation(N_TRAIN)
    X_shuf, y_shuf = X_train_np[idx], y_train_np[idx]

    epoch_loss = 0
    for i in range(0, N_TRAIN, BATCH):
        xb, yb = X_shuf[i:i+BATCH], y_shuf[i:i+BATCH]
        model_np.forward(xb)
        epoch_loss += model_np.loss(yb)
        model_np.backward(yb)

    n_batches = N_TRAIN // BATCH
    avg_loss = epoch_loss / n_batches
    acc = model_np.accuracy(X_test_np, y_test_np)
    np_losses.append(avg_loss)
    np_accs.append(acc)

    wandb.log({'numpy/loss': avg_loss, 'numpy/test_acc': acc}, step=epoch)
    if epoch % 5 == 0:
        print(f'[NumPy] Epoch {epoch:3d} | loss: {avg_loss:.4f} | test acc: {acc*100:.1f}%')

print(f'\n[NumPy] 최종 test acc: {np_accs[-1]*100:.1f}%')

---
## Part 2 — PyTorch MLP

NumPy 구현과 **동일한 구조**이지만, 역전파는 autograd에 맡깁니다.

```
NumPy           →   PyTorch
────────────────────────────
X @ W + b       →   nn.Linear(784, 256)
np.maximum(0,z) →   F.relu(z)
수동 역전파     →   loss.backward()
p.data -= lr*g  →   optimizer.step()
```

In [ ]:
class PyTorchMLP(nn.Module):
    """
    PyTorch nn.Module 버전.
    내부 구조는 NumPy 버전과 완전히 동일.
    nn.Linear = 행렬 곱셈(W) + 브로드캐스팅(b) 를 하나로 묶은 것.
    """
    def __init__(self, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(784, 256),   # Z1 = X @ W1 + b1  (과제1 내용!)
            nn.BatchNorm1d(256),   # 학습 안정화
            nn.ReLU(),             # A1 = relu(Z1)
            nn.Dropout(dropout),   # 과적합 방지

            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(128, 10),    # 출력층 (10 클래스)
        )

    def forward(self, x):
        x = x.view(x.size(0), -1)  # (N,1,28,28) → (N,784) flatten
        return self.net(x)


# 데이터 로더
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=512, shuffle=False, num_workers=2)

# 모델 + 옵티마이저
model_pt = PyTorchMLP().to(DEVICE)
optimizer = torch.optim.AdamW(   # ← AdamW! (방금 배운 것)
    model_pt.parameters(),
    lr=1e-3,
    weight_decay=0.01             # 파라미터에 직접 감쇠 적용
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=10           # LLM 표준 패턴
)
criterion = nn.CrossEntropyLoss()

n_params = sum(p.numel() for p in model_pt.parameters())
print(f'모델 파라미터 수: {n_params:,}')

In [ ]:
# PyTorch 학습 루프
EPOCHS_PT = 10
pt_losses, pt_accs = [], []

for epoch in range(EPOCHS_PT):
    # ── 학습 ──────────────────────────────────────────────────────────
    model_pt.train()
    train_loss = 0
    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)

        optimizer.zero_grad()           # ∇ 초기화 (안 하면 누적!)
        pred  = model_pt(x)             # 순전파
        loss  = criterion(pred, y)      # CrossEntropy loss
        loss.backward()                 # 역전파: 모든 ∂L/∂wᵢ 계산
        torch.nn.utils.clip_grad_norm_( # 그래디언트 폭발 방지
            model_pt.parameters(), 1.0
        )
        optimizer.step()                # AdamW 업데이트
        train_loss += loss.item()

    scheduler.step()                    # lr 코사인 감소

    # ── 평가 ──────────────────────────────────────────────────────────
    model_pt.eval()
    correct = 0
    with torch.no_grad():               # 평가 시 그래디언트 불필요
        for x, y in test_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            correct += (model_pt(x).argmax(1) == y).sum().item()

    avg_loss = train_loss / len(train_loader)
    acc = correct / len(test_dataset)
    pt_losses.append(avg_loss)
    pt_accs.append(acc)

    wandb.log({
        'pytorch/train_loss': avg_loss,
        'pytorch/test_acc':   acc,
        'pytorch/lr':         scheduler.get_last_lr()[0]
    }, step=epoch)
    print(f'[PyTorch] Epoch {epoch+1:3d} | loss: {avg_loss:.4f} | acc: {acc*100:.2f}% | lr: {scheduler.get_last_lr()[0]:.5f}')

print(f'\n[PyTorch] 최종 test acc: {pt_accs[-1]*100:.2f}%')

---
## Part 3 — PyTorch Lightning MLP

학습 루프를 Lightning이 처리합니다. 코드는 더 짧아지고, 실험 재현성은 높아집니다.

```
PyTorch         →   Lightning
────────────────────────────────────────
학습 루프 직접  →   Trainer.fit()이 처리
model.train()   →   training_step() 정의
model.eval()    →   validation_step() 정의
optimizer 정의  →   configure_optimizers() 정의
```

In [ ]:
class LightningMLP(pl.LightningModule):
    """
    PyTorch Lightning 버전.
    training_step / validation_step / configure_optimizers 만 정의하면
    나머지는 Trainer가 자동으로 처리합니다.
    """
    def __init__(self, lr=1e-3, weight_decay=0.01, dropout=0.3):
        super().__init__()
        self.save_hyperparameters()     # hparams 자동 저장 (wandb 연동)
        self.net = nn.Sequential(
            nn.Linear(784, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        return self.net(x.view(x.size(0), -1))

    def training_step(self, batch, batch_idx):
        x, y = batch
        loss = F.cross_entropy(self(x), y)
        acc  = (self(x).argmax(1) == y).float().mean()
        self.log('train/loss', loss, prog_bar=True)
        self.log('train/acc',  acc,  prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        loss = F.cross_entropy(self(x), y)
        acc  = (self(x).argmax(1) == y).float().mean()
        self.log('val/loss', loss, prog_bar=True)
        self.log('val/acc',  acc,  prog_bar=True)

    def configure_optimizers(self):
        # AdamW + CosineAnnealingLR — LLM 학습 표준 패턴 그대로
        opt = torch.optim.AdamW(
            self.parameters(),
            lr=self.hparams.lr,
            weight_decay=self.hparams.weight_decay
        )
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=10)
        return {'optimizer': opt, 'lr_scheduler': sch}


print('LightningMLP 정의 완료 ✓')

In [ ]:
from pytorch_lightning.loggers import WandbLogger
from pytorch_lightning.callbacks import (
    ModelCheckpoint, EarlyStopping, LearningRateMonitor
)

# 콜백 설정
callbacks = [
    ModelCheckpoint(
        monitor='val/acc', mode='max',   # val acc 최고일 때 저장
        filename='best-{epoch:02d}-{val/acc:.3f}'
    ),
    EarlyStopping(
        monitor='val/loss', patience=5,  # 5 에폭 개선 없으면 조기 종료
        mode='min'
    ),
    LearningRateMonitor(logging_interval='epoch'),
]

# wandb 로거
wandb_logger = WandbLogger(
    project='portfolio-tsuruoka-lab',
    name='phase0-03-lightning'
)

# 모델 + Trainer
model_lg = LightningMLP(lr=1e-3, weight_decay=0.01, dropout=0.3)
trainer  = pl.Trainer(
    max_epochs=10,
    accelerator='auto',
    logger=wandb_logger,
    callbacks=callbacks,
    enable_progress_bar=True,
)

# 학습 — 루프는 trainer.fit 한 줄!
trainer.fit(model_lg, train_loader, test_loader)

# 최종 평가
results = trainer.test(model_lg, test_loader)
print(f'\n[Lightning] 최종 test acc: {results[0]["val/acc"]*100:.2f}%')

---
## 4. 하이퍼파라미터 스윕 — wandb로 최적 설정 탐색

학습률, weight_decay, dropout을 여러 조합으로 실험하고 최적을 찾습니다.
이것이 실제 논문 실험 방식입니다.

In [ ]:
# 간단한 그리드 서치 (wandb 스윕 대신 직접 루프)
# 실제 대규모 스윕은 wandb.sweep() API 사용

sweep_configs = [
    {'lr': 1e-3, 'wd': 0.01,  'dropout': 0.2},
    {'lr': 1e-3, 'wd': 0.01,  'dropout': 0.4},
    {'lr': 5e-4, 'wd': 0.001, 'dropout': 0.3},
    {'lr': 2e-3, 'wd': 0.1,   'dropout': 0.3},
]

sweep_results = []

for cfg in sweep_configs:
    print(f"\n실험: lr={cfg['lr']}, wd={cfg['wd']}, dropout={cfg['dropout']}")

    m = LightningMLP(
        lr=cfg['lr'],
        weight_decay=cfg['wd'],
        dropout=cfg['dropout']
    )
    t = pl.Trainer(
        max_epochs=5,          # 스윕은 빠르게
        accelerator='auto',
        enable_progress_bar=False,
        logger=False,
        enable_checkpointing=False,
    )
    t.fit(m, train_loader, test_loader)
    res = t.test(m, test_loader, verbose=False)
    acc = res[0].get('val/acc', 0)
    sweep_results.append({**cfg, 'acc': acc})
    wandb.log({f'sweep/acc': acc, **{f'sweep/{k}':v for k,v in cfg.items()}})
    print(f'  → acc: {acc*100:.2f}%')

best = max(sweep_results, key=lambda x: x['acc'])
print(f'\n최적 설정: {best}')

---
## 5. 세 구현 결과 비교 및 wandb 최종 기록

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# NumPy 학습 곡선
axes[0].plot(np_losses, color='#7F77DD', label='NumPy SGD', linewidth=2)
axes[0].set_title('NumPy Loss'); axes[0].set_xlabel('Epoch')
axes[0].legend(); axes[0].grid(alpha=.3)

# PyTorch 정확도
axes[1].plot([a*100 for a in pt_accs], color='#D85A30', linewidth=2)
axes[1].set_title('PyTorch Test Accuracy (%)'); axes[1].set_xlabel('Epoch')
axes[1].set_ylim(90, 100); axes[1].grid(alpha=.3)

# 스윕 결과
labels = [f"lr={c['lr']}\nwd={c['wd']}\ndo={c['dropout']}" for c in sweep_results]
accs   = [c['acc']*100 for c in sweep_results]
colors = ['#D85A30' if a==max(accs) else '#B4B2A9' for a in accs]
axes[2].bar(range(len(accs)), accs, color=colors)
axes[2].set_xticks(range(len(accs))); axes[2].set_xticklabels(labels, fontsize=7)
axes[2].set_title('Hyperparameter Sweep'); axes[2].set_ylabel('Test Acc (%)')
axes[2].set_ylim(min(accs)-2, 100); axes[2].grid(axis='y', alpha=.3)

plt.tight_layout()
wandb.log({'results/comparison': wandb.Image(fig)})
plt.savefig('mlp_mnist_results.png', dpi=150, bbox_inches='tight')
plt.show()

print('='*55)
print('Phase 0 — 구현 과제 3 결과 요약')
print('='*55)
print(f'NumPy   MLP | 최종 test acc: {np_accs[-1]*100:.1f}%')
print(f'PyTorch MLP | 최종 test acc: {pt_accs[-1]*100:.2f}%')
print(f'최적 스윕   | acc: {best["acc"]*100:.2f}%  {best}')

wandb.summary.update({
    'numpy_final_acc':   np_accs[-1],
    'pytorch_final_acc': pt_accs[-1],
    'best_sweep_acc':    best['acc'],
    'best_config':       str(best),
})
wandb.finish()
print('\nwandb run 완료 ✓  ← README에 링크 추가하세요!')

---
## 6. 체크리스트 & Phase 0 완료

### 구현 과제 3 완료 확인
- [ ] NumPy MLP: 행렬 곱셈 + 역전파 직접 구현, ~95% 이상
- [ ] PyTorch MLP: AdamW + CosineAnnealingLR + grad clipping
- [ ] Lightning MLP: training_step / configure_optimizers 정의
- [ ] 하이퍼파라미터 스윕 4개 이상 실험
- [ ] wandb 학습 곡선 + 스윕 결과 업로드
- [ ] GitHub commit

### Commit 메시지
```
phase0/03: MLP MNIST — numpy → pytorch → lightning

- numpy mlp: matmul + manual backprop, ~95% acc
- pytorch mlp: AdamW + cosine lr + grad clipping
- lightning mlp: training_step + configure_optimizers
- sweep: 4 configs, best acc logged to wandb
- wandb: all curves + comparison chart uploaded
```

### Phase 0 전체 연결
| 과제 | 핵심 구현 | Phase 1+에서 다시 만나는 곳 |
|---|---|---|
| 01 numpy linear algebra | `A @ B`, 브로드캐스팅, 코사인 유사도 | Attention 내부 |
| 02 micrograd | `Value._backward()`, 위상 정렬 | `loss.backward()` 원리 |
| 03 mlp mnist | NumPy → PyTorch → Lightning | Phase 1 ResNet, Phase 2 Transformer |

### 🎉 Phase 0 완료!

**다음: Phase 1 — Deep Learning 기초**
- `01-resnet-from-scratch`: CNN + 잔차 연결
- `02-batch-norm`: 학습 안정화의 핵심
- 목표: ImageNet 분류 구현